In [15]:
!pip install -q sentence-transformers pandas scikit-learn

In [16]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [17]:
sdgs = {
    "SDG 1": "No poverty",
    "SDG 2": "Zero hunger",
    "SDG 3": "Good health and well-being",
    "SDG 4": "Quality education",
    "SDG 5": "Gender equality",
    "SDG 6": "Clean water and sanitation",
    "SDG 7": "Affordable and clean energy",
}

In [18]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
print(predict_sdg_top3("A mobile app that improves rural healthcare and education access"))

['SDG 4', 'SDG 1', 'SDG 7']


In [20]:
sdg_texts = list(sdgs.values())
sdg_embeddings = model.encode(sdg_texts)

In [21]:
def _predict_sdg_with_details(text):
    text_embedding = model.encode(text)
    similarities = cosine_similarity([text_embedding], sdg_embeddings)[0]

    sdg_preds = []
    for i, sim in enumerate(similarities):
        sdg_code = list(sdgs.keys())[i]
        sdg_description = sdgs[sdg_code]
        sdg_preds.append((sdg_code, sdg_description, sim))

    # Sort by similarity in descending order and return top 3
    return sorted(sdg_preds, key=lambda x: x[2], reverse=True)[:3]

In [22]:
def predict_sdg(text):
    preds_with_details = _predict_sdg_with_details(text)
    if preds_with_details:
        top_pred = preds_with_details[0]
        return (top_pred[0], top_pred[1]) # Return (sdg_code, sdg_description)
    return (None, None)

def predict_sdg_top3(text):
    preds_with_details = _predict_sdg_with_details(text)
    return [p[0] for p in preds_with_details] # Return list of sdg_codes

In [23]:
def predict_sdg(text):
    preds_with_details = _predict_sdg_with_details(text)
    if preds_with_details:
        top_pred = preds_with_details[0]
        return (top_pred[0], top_pred[1]) # Return (sdg_code, sdg_description)
    return (None, None)

def predict_sdg_top3(text):
    preds_with_details = _predict_sdg_with_details(text)
    return [p[0] for p in preds_with_details] # Return list of sdg_codes

In [30]:
def evaluate(df_to_evaluate):
    correct = 0
    total_predictions = len(df_to_evaluate)

    print(f"Evaluating {total_predictions} projects...")
    for _, row in df_to_evaluate.iterrows():
        pred_code, _ = predict_sdg(row["project"])

        # print("Project:", row["project"])
        # print("Predicted:", pred_code)
        # print("Actual:", row["actual_sdg"])
        # print("-----")

        if pred_code == row["actual_sdg"]:
            correct += 1

    accuracy = correct / total_predictions
    print(f"Accuracy: {accuracy:.4f} ({correct}/{total_predictions})\n")

In [24]:
test_text = "A mobile app that helps rural students access education"

print(predict_sdg(test_text))

('SDG 4', 'Quality education')


In [31]:
evaluate(df)

Evaluating 4 projects...
Accuracy: 1.0000 (4/4)



In [26]:
data = {
    "project": [
        "Mobile health app for villages",
        "Online education platform for students",
        "Clean drinking water system",
        "Solar energy for rural homes"
    ],
    "actual_sdg": [
        "SDG 3",
        "SDG 4",
        "SDG 6",
        "SDG 7"
    ]
}

df = pd.DataFrame(data)
df

,project,actual_sdg
0,Mobile health app for villages,SDG 3
1,Online education platform for students,SDG 4
2,Clean drinking water system,SDG 6
3,Solar energy for rural homes,SDG 7


In [28]:
import numpy as np

# More comprehensive DPG-based dataset (simulated)
dpg_data = {
    "project": [
        "Developing open-source software for sustainable agriculture",
        "Providing digital literacy training in underserved communities",
        "Implementing blockchain solutions for transparent supply chains",
        "Creating AI tools for early disease detection in rural areas",
        "Open data platform for climate change research",
        "Digital identity system for refugees",
        "Renewable energy microgrid for remote villages",
        "Affordable assistive technology for people with disabilities",
        "Online platform connecting small farmers to markets",
        "Satellite imagery analysis for deforestation monitoring"
    ],
    "actual_sdg": [
        "SDG 2", # Zero hunger
        "SDG 4", # Quality education
        "SDG 9", # Industry, Innovation, and Infrastructure (proxy for transparent supply chains)
        "SDG 3", # Good health and well-being
        "SDG 13", # Climate Action (proxy for climate change research)
        "SDG 16", # Peace, Justice, and Strong Institutions (proxy for digital identity)
        "SDG 7", # Affordable and clean energy
        "SDG 10", # Reduced inequalities (proxy for assistive technology)
        "SDG 2", # Zero hunger
        "SDG 15"  # Life on Land
    ]
}

# Expand SDG descriptions for the simulated dataset if new SDGs are introduced
# (Assuming SDG 9, 10, 13, 15, 16 might be new for this expanded dataset)
sdgs.update({
    "SDG 8": "Decent work and economic growth",
    "SDG 9": "Industry, innovation and infrastructure",
    "SDG 10": "Reduced inequalities",
    "SDG 11": "Sustainable cities and communities",
    "SDG 12": "Responsible consumption and production",
    "SDG 13": "Climate action",
    "SDG 14": "Life below water",
    "SDG 15": "Life on land",
    "SDG 16": "Peace, justice and strong institutions",
    "SDG 17": "Partnerships for the goals"
})

# Re-encode sdg_embeddings with the expanded list of SDGs
sdg_texts = list(sdgs.values())
sdg_embeddings = model.encode(sdg_texts)

df_dpg = pd.DataFrame(dpg_data)
df_dpg

,project,actual_sdg
0,Developing open-source software for sustainabl...,SDG 2
1,Providing digital literacy training in underse...,SDG 4
2,Implementing blockchain solutions for transpar...,SDG 9
3,Creating AI tools for early disease detection ...,SDG 3
4,Open data platform for climate change research,SDG 13
5,Digital identity system for refugees,SDG 16
6,Renewable energy microgrid for remote villages,SDG 7
7,Affordable assistive technology for people wit...,SDG 10
8,Online platform connecting small farmers to ma...,SDG 2
9,Satellite imagery analysis for deforestation m...,SDG 15


In [32]:
# Evaluate with the new DPG dataset
print("\n--- Evaluation on DPG Dataset ---")
evaluate(df_dpg)


--- Evaluation on DPG Dataset ---
Evaluating 10 projects...
Accuracy: 0.5000 (5/10)



In [33]:
import numpy as np

# More comprehensive DPG-based dataset (simulated)
dpg_data = {
    "project": [
        "Developing open-source software for sustainable agriculture",
        "Providing digital literacy training in underserved communities",
        "Implementing blockchain solutions for transparent supply chains",
        "Creating AI tools for early disease detection in rural areas",
        "Open data platform for climate change research",
        "Digital identity system for refugees",
        "Renewable energy microgrid for remote villages",
        "Affordable assistive technology for people with disabilities",
        "Online platform connecting small farmers to markets",
        "Satellite imagery analysis for deforestation monitoring"
    ],
    "actual_sdg": [
        "SDG 2", # Zero hunger
        "SDG 4", # Quality education
        "SDG 9", # Industry, Innovation, and Infrastructure (proxy for transparent supply chains)
        "SDG 3", # Good health and well-being
        "SDG 13", # Climate Action (proxy for climate change research)
        "SDG 16", # Peace, Justice, and Strong Institutions (proxy for digital identity)
        "SDG 7", # Affordable and clean energy
        "SDG 10", # Reduced inequalities (proxy for assistive technology)
        "SDG 2", # Zero hunger
        "SDG 15"  # Life on Land
    ]
}

# Expand SDG descriptions for the simulated dataset if new SDGs are introduced
# (Assuming SDG 9, 10, 13, 15, 16 might be new for this expanded dataset)
sdgs.update({
    "SDG 8": "Decent work and economic growth",
    "SDG 9": "Industry, innovation and infrastructure",
    "SDG 10": "Reduced inequalities",
    "SDG 11": "Sustainable cities and communities",
    "SDG 12": "Responsible consumption and production",
    "SDG 13": "Climate action",
    "SDG 14": "Life below water",
    "SDG 15": "Life on land",
    "SDG 16": "Peace, justice and strong institutions",
    "SDG 17": "Partnerships for the goals"
})

# Re-encode sdg_embeddings with the expanded list of SDGs
sdg_texts = list(sdgs.values())
sdg_embeddings = model.encode(sdg_texts)

df_dpg = pd.DataFrame(dpg_data)
df_dpg

,project,actual_sdg
0,Developing open-source software for sustainabl...,SDG 2
1,Providing digital literacy training in underse...,SDG 4
2,Implementing blockchain solutions for transpar...,SDG 9
3,Creating AI tools for early disease detection ...,SDG 3
4,Open data platform for climate change research,SDG 13
5,Digital identity system for refugees,SDG 16
6,Renewable energy microgrid for remote villages,SDG 7
7,Affordable assistive technology for people wit...,SDG 10
8,Online platform connecting small farmers to ma...,SDG 2
9,Satellite imagery analysis for deforestation m...,SDG 15


In [34]:
# Evaluate with the new DPG dataset
print("\n--- Evaluation on DPG Dataset ---")
evaluate(df_dpg)


--- Evaluation on DPG Dataset ---
Evaluating 10 projects...
Accuracy: 0.5000 (5/10)



In [35]:
from ipywidgets import Textarea, Button, VBox, Output, Layout
from IPython.display import display

print("### SDG Prediction Interface ###\n")

project_input = Textarea(
    value='',
    placeholder='Enter project description here...',
    description='Project:',
    disabled=False,
    layout=Layout(width='auto', height='100px')
)

predict_button = Button(description="Predict SDG")
output_area = Output()

def on_button_click(b):
    with output_area:
        output_area.clear_output()
        project_description = project_input.value
        if project_description:
            print(f"Analyzing project: '{project_description}'")
            predictions = _predict_sdg_with_details(project_description)
            print("\nPredicted Top 3 SDGs:")
            for sdg_code, sdg_desc, score in predictions:
                print(f"  - {sdg_code}: {sdg_desc} (Similarity: {score:.4f})")
        else:
            print("Please enter a project description.")

predict_button.on_click(on_button_click)

display(VBox([project_input, predict_button, output_area]))

### SDG Prediction Interface ###



In [36]:
from ipywidgets import Textarea, Button, VBox, Output, Layout
from IPython.display import display

print("### SDG Prediction Interface ###\n")

project_input = Textarea(
    value='',
    placeholder='Enter project description here...',
    description='Project:',
    disabled=False,
    layout=Layout(width='auto', height='100px')
)

predict_button = Button(description="Predict SDG")
output_area = Output()

def on_button_click(b):
    with output_area:
        output_area.clear_output()
        project_description = project_input.value
        if project_description:
            print(f"Analyzing project: '{project_description}'")
            predictions = _predict_sdg_with_details(project_description)
            print("\nPredicted Top 3 SDGs:")
            for sdg_code, sdg_desc, score in predictions:
                print(f"  - {sdg_code}: {sdg_desc} (Similarity: {score:.4f})")
        else:
            print("Please enter a project description.")

predict_button.on_click(on_button_click)

display(VBox([project_input, predict_button, output_area]))

### SDG Prediction Interface ###



In [37]:
correct = 0

for _, row in df.iterrows():
    pred, _ = predict_sdg(row["project"])

    print("Project:", row["project"])
    print("Predicted:", pred)
    print("Actual:", row["actual_sdg"])
    print("---")

    if pred == row["actual_sdg"]:
        correct += 1

accuracy = correct / len(df)

print("Accuracy:", accuracy)

Project: Mobile health app for villages
Predicted: SDG 3
Actual: SDG 3
---
Project: Online education platform for students
Predicted: SDG 4
Actual: SDG 4
---
Project: Clean drinking water system
Predicted: SDG 6
Actual: SDG 6
---
Project: Solar energy for rural homes
Predicted: SDG 7
Actual: SDG 7
---
Accuracy: 1.0


In [39]:
# Install nbformat if not already installed
!pip install -q nbformat

import nbformat
import os
import glob # Import glob for pattern matching

# This function will attempt to clear outputs and widget metadata
def clean_notebook_for_git(notebook_file_path):
    notebook_content = None
    try:
        # First, try the provided path directly
        with open(notebook_file_path, 'r', encoding='utf-8') as f:
            notebook_content = nbformat.read(f, as_version=4)

    except FileNotFoundError:
        # If not found, try to find a single .ipynb file in the current directory
        ipynb_files = glob.glob('*.ipynb')
        if len(ipynb_files) == 1:
            detected_notebook_path = ipynb_files[0]
            print(f"Detected notebook file: '{detected_notebook_path}'. Attempting to use this.")
            with open(detected_notebook_path, 'r', encoding='utf-8') as f:
                notebook_content = nbformat.read(f, as_version=4)
            notebook_file_path = detected_notebook_path # Update path for writing
        elif len(ipynb_files) > 1:
            print(f"Error: Multiple .ipynb files found in the current directory: {ipynb_files}.")
            print("Please update 'NOTEBOOK_PATH' to the correct notebook name manually.")
            return
        else:
            print(f"Error: No .ipynb file found in the current directory.")
            print("Please ensure you are running this in a Colab notebook and 'NOTEBOOK_PATH' is correct.")
            return
    except Exception as e:
        print(f"An error occurred during notebook file reading: {e}")
        return

    if notebook_content:
        try:
            # Clear outputs from all cells
            for cell in notebook_content.cells:
                if hasattr(cell, 'outputs'):
                    cell.outputs = []
                if hasattr(cell, 'execution_count'):
                    cell.execution_count = None

                # Remove widget metadata from cell metadata
                if 'metadata' in cell and 'widgets' in cell['metadata']:
                    del cell['metadata']['widgets']

            # Remove top-level notebook widget metadata
            if 'metadata' in notebook_content and 'widgets' in notebook_content['metadata']:
                del notebook_content['metadata']['widgets']

            with open(notebook_file_path, 'w', encoding='utf-8') as f:
                nbformat.write(notebook_content, f)
            print(f"Notebook '{notebook_file_path}' cleaned successfully for Git.")
            print("Remember to save your notebook manually in Colab (File -> Save) after running this cell,")
            print("then download it and add it to your Git repository.")

        except Exception as e:
            print(f"An error occurred during notebook cleaning: {e}")

# Call the cleaning function
# NOTE: You might need to manually refresh the Colab page or download the notebook again
# after running this to see the changes reflected in the downloaded .ipynb file.
# Let's try to make this dynamic or fallback more gracefully.

# NOTE: In Colab, the primary notebook is often the only .ipynb file in /content/.
# We will leverage this. Set an initial placeholder and let the function detect.
NOTEBOOK_PATH_PLACEHOLDER = 'Untitled.ipynb'
clean_notebook_for_git(NOTEBOOK_PATH_PLACEHOLDER)


Error: No .ipynb file found in the current directory.
Please ensure you are running this in a Colab notebook and 'NOTEBOOK_PATH' is correct.
